<a href="https://colab.research.google.com/github/JosephAFerguson/-UserInterface-Proj2/blob/main/DeepLearningJF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects
import json
from datetime import datetime, timedelta

In [56]:
class CryptoEndpoint:
    listingsEndpoint = "https://sandbox-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
    latestQuotes = "https://sandbox-api.coinmarketcap.com/v2/cryptocurrency/quotes/latest"
    historicalQuotes = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/quotes/historical"

    def __init__(self, apikey) -> None:
        self.headers = {
            'Accepts': 'application/json',
            'X-CMC_PRO_API_KEY': apikey,
        }
        self.coinsIdentifiers = {}

    def GetCoinIdentifiers(self):
        session = Session()
        session.headers.update(self.headers)

        response = session.get(url=self.listingsEndpoint, params={"limit": 50})
        data = json.loads(response.text)

        for coin in data.get("data", []):
            self.coinsIdentifiers[coin["symbol"]] = coin["id"]

        print(f"Loaded {len(self.coinsIdentifiers)} coins.")
        return self.coinsIdentifiers

    def GetCoinLatestPrices(self):
        if not self.coinsIdentifiers:
            print("No coins loaded yet. Run GetCoinIdentifiers() first.")
            return

        session = Session()
        session.headers.update(self.headers)

        ids = ",".join(str(v) for v in self.coinsIdentifiers.values())

        response = session.get(url=self.latestQuotes, params={"id": ids})
        data = json.loads(response.text)

        prices = {}
        for coin_id, info in data.get("data", {}).items():
            quote = info["quote"]["USD"]["price"]
            prices[info["symbol"]] = quote

        return prices

    def GetCoinHistoricalPrices(self, days=7):
        if not self.coinsIdentifiers:
            print("No coins loaded yet. Run GetCoinIdentifiers() first.")
            return

        session = Session()
        session.headers.update(self.headers)

        end_time = datetime.utcnow()
        start_time = end_time - timedelta(days=days)

        prices = {}

        for symbol, coin_id in self.coinsIdentifiers.items():
          params = {
              "id": coin_id,
              "time_start": start_time.isoformat(),
              "time_end": end_time.isoformat(),
              "interval": "24h",
          }

          response = session.get(url=self.historicalQuotes, params=params)
          data = json.loads(response.text)
          print(data)
          coin_data = data.get("data", {})
          if not coin_data or "quotes" not in coin_data:
              continue

          history = []

          for quote in coin_data["quotes"]:
              date = quote.get("timestamp") or quote.get("time_open")
              price = quote["quote"]["USD"]["price"]
              history.append((date, price))

          prices[symbol] = history

        return prices

In [59]:
ce = CryptoEndpoint(input("Enter API-KEY"))
ce.GetCoinIdentifiers()
prices = ce.GetCoinHistoricalPrices()
print(prices)

Enter API-KEYa76bd6fc-2b66-4a25-843f-321de-f3437bd
Loaded 10 coins.
{'status': {'timestamp': '2025-10-27T01:27:14.399Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.457Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}


/tmp/ipython-input-345299755.py:54: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


{'status': {'timestamp': '2025-10-27T01:27:14.513Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.570Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.640Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.695Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.751Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.809Z', 'error_code': 1001, 'error_message': 'This API Key is invalid.', 'elapsed': 0, 'credit_count': 0}}
{'status': {'timestamp': '2025-10-27T01:27:14.864Z', 'error_code': 1001, 'error_message'